## Лабораторная работа № 1 
## Выполнение разведочного анализа больших данных с использованием фреймворка Apache Spark

### Часть 1

В данной части работы рассмотрены:
* загрузка данных из HDFS;
* базовые преобразования данных;
* загрузка преобразованных данных в таблицу `Apache Iceberg`.

Подключим необходимые библиотеки.

In [ ]:
import os
from pyspark.sql import SparkSession, DataFrame
from pyspark import SparkConf
from pyspark.sql.functions import (
    col,
    lit,
    when,
    split,
    to_date,
    regexp_extract,
    min as min_,
    max as max_
)

Сформируем объект конфигурации для `Apache Spark`, указав необходимые параметры.

In [ ]:
def create_spark_configuration() -> SparkConf:  # объявляем функцию без параметров; "-> SparkConf" означает, что она вернёт объект конфигурации
    """
    Создает и конфигурирует экземпляр SparkConf для приложения Spark.

    Returns:
        SparkConf: Настроенный экземпляр SparkConf.
    """
    # Получаем имя пользователя
    user_name = os.getenv("USER")  # читаем имя пользователя из переменной окружения USER (у вас user0); оно нужно для пути к хранилищу таблиц

    conf = SparkConf()  # создаём пустой объект настроек, который дальше заполним
    conf.setAppName("lab 1 Flights")  # имя приложения: отображается в интерфейсе YARN и Spark UI, помогает найти свою задачу среди других
    conf.setMaster("yarn")  # менеджер ресурсов: "yarn" = вычисления идут на кластере Hadoop, а не на одной машине
    conf.set("spark.submit.deployMode", "client")  # режим "client": управляющая программа (драйвер) работает на узле с вашим ноутбуком, а вычисления выполняют исполнители на кластере
    conf.set("spark.executor.memory", "12g")  # оперативная память каждого исполнителя: 12 ГБ
    conf.set("spark.executor.cores", "8")  # число ядер процессора у каждого исполнителя: 8
    conf.set("spark.executor.instances", "2")  # число исполнителей: 2 (вместе 2 x 8 = 16 параллельных задач)
    conf.set("spark.driver.memory", "4g")  # память драйвера: 4 ГБ; он принимает результаты collect() и toPandas(), поэтому не стоит просить у него слишком много данных
    conf.set("spark.driver.cores", "2")  # число ядер драйвера: 2
    conf.set("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.0")  # подключаем библиотеку Apache Iceberg 1.6.0 для Spark 3.5 и Scala 2.12; она скачивается при создании сессии
    conf.set("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")  # включаем расширение Spark SQL, которое добавляет поддержку команд и таблиц Iceberg
    conf.set("spark.sql.catalog.spark_catalog", "org.apache.iceberg.spark.SparkCatalog")  # делаем каталог Spark по умолчанию каталогом Iceberg, чтобы создавать и читать таблицы этого формата
    conf.set("spark.sql.catalog.spark_catalog.type", "hadoop")  # тип каталога "hadoop": метаданные таблиц хранятся прямо в файловой системе HDFS, отдельный сервер не нужен
    conf.set("spark.sql.catalog.spark_catalog.warehouse", f"hdfs:///user/{user_name}/warehouse")  # папка в HDFS, где лежат таблицы; f-строка подставляет ваше имя (hdfs:///user/user0/warehouse)
    conf.set("spark.sql.catalog.spark_catalog.io-impl", "org.apache.iceberg.hadoop.HadoopFileIO")  # класс, через который Iceberg читает и пишет файлы в HDFS

    return conf  # возвращаем готовую конфигурацию, которую потом передаём в SparkSession.builder.config(conf=conf)

Создаём сам объект конфигурации.

In [3]:
conf = create_spark_configuration()

Создаём и выводим на экран сессию `Apache Spark`. В процессе создания сессии происходит подключение к кластеру `Apache Hadoop`, что может занять некоторое время.

In [4]:
spark = SparkSession.builder.config(conf=conf).getOrCreate()
spark

:: loading settings :: url = jar:file:/opt/spark-3.5.2-bin-hadoop3/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/user0/.ivy2/cache
The jars for the packages stored in: /home/user0/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-f7dd6279-84cb-45f3-996c-3f092030db95;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.6.0 in central
:: resolution report :: resolve 619ms :: artifacts dl 26ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.6.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   0   |   0   |   0   ||   1   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spa

Для исследования будем использовать датасет `"Flight Prices"`, расположенный на платформе `Kaggle` по адресу https://www.kaggle.com/datasets/dilwong/flightprices.

Датасет включает в себя информацию о десятках миллионов вариантов авиаперелётов (билетов в одну сторону) между крупнейшими аэропортами США, собранную с сервиса `Expedia` в период с апреля по октябрь 2022 года. Каждая строка описывает один вариант перелёта (маршрут с возможными пересадками), найденный в конкретный день поиска, вместе с его стоимостью. 

Данный датасет загружен в `HDFS` по адресу: `hdfs:///user/user0/datasets/itineraries_apr16_may17.csv`

Указываем путь в `HDFS` для файла с данными.

In [ ]:
path = "hdfs:///user/user0/datasets/itineraries_apr16_may17.csv"

Заполняем датафрейм данными из файла.

In [6]:
df = (spark.read.format("csv")
      .option("header", "true")
      .load(path)
)

Исходный датасет очень велик (десятки миллионов строк, десятки гигабайт, данные за период с 2022-04-16 по 2022-10-05). Многократные проходы по такому объему данных (разведочный анализ, а в дальнейшем и обучение моделей с кросс-валидацией) потребуют значительного расчетного времени. Поэтому в учебных целях мы обрезали файл ещё до загрузки на кластер: с помощью скрипта на локальном компьютере в файле оставлены только строки, дата поиска (`searchDate`) которых лежит в интервале с 2022-04-16 по 2022-05-17 включительно (один месяц). Тем самым в `HDFS` и далее в `Spark` попадают только нужные данные.

Все дальнейшие действия выполняются уже с этим обрезанным файлом.

Выводим фрагмент датафрейма на экран. Длинные строковые значения обрезаем для удобства чтения.

In [7]:
df.show(5, truncate=30)

26/10/17 18:17:29 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+------------------------------+----------+----------+---------------+------------------+-------------+--------------+-----------+--------------+------------+---------+--------+---------+--------------+-------------------+---------------------------------+-----------------------------+-------------------------------+-----------------------------+--------------------------+----------------------------+-------------------+-------------------+----------------------------+-------------------------+----------------+-----------------+
|                         legId|searchDate|flightDate|startingAirport|destinationAirport|fareBasisCode|travelDuration|elapsedDays|isBasicEconomy|isRefundable|isNonStop|baseFare|totalFare|seatsRemaining|totalTravelDistance|segmentsDepartureTimeEpochSeconds|     segmentsDepartureTimeRaw|segmentsArrivalTimeEpochSeconds|       segmentsArrivalTimeRaw|segmentsArrivalAirportCode|segmentsDepartureAirportCode|segmentsAirlineName|segmentsAirlineCode|segmentsEquipmentDesc

Очевидно, что в целях сохранения ясности изложения и сокращения расчетного времени имеет смысл рассматривать не все столбцы датасета. Оставим следующие колонки, удалив остальные:

| Название столбца      | Расшифровка |
| --------------------- | ------------- |
| legId                 | Идентификатор перелёта |
| searchDate            | Дата, в которую запись была получена с сервиса Expedia |
| flightDate            | Дата вылета |
| startingAirport       | IATA-код аэропорта вылета |
| destinationAirport    | IATA-код аэропорта прилёта |
| travelDuration        | Общая длительность поездки (включая пересадки) в формате ISO 8601, например `PT2H29M` |
| elapsedDays           | Количество суток, прошедших между вылетом и прилётом (обычно 0) |
| isBasicEconomy        | Является ли билет билетом базового экономического тарифа |
| isRefundable          | Возвращаемый ли билет |
| isNonStop             | Является ли перелёт прямым (без пересадок) |
| totalFare             | Итоговая цена билета в долларах США, включая налоги и сборы |
| seatsRemaining        | Количество оставшихся мест |
| totalTravelDistance   | Общая дальность перелёта в милях (иногда отсутствует) |
| segmentsDepartureTimeRaw | Время вылета для каждого сегмента перелёта (ISO 8601); значения для сегментов разделены `\|\|` |
| segmentsAirlineName   | Название авиакомпании для каждого сегмента; значения разделены `\|\|` |
| segmentsCabinCode     | Класс обслуживания для каждого сегмента (например, `coach`); значения разделены `\|\|` |

Остальные столбцы удаляются по следующим причинам:

| Удаляемый столбец | Причина |
| ----------------- | ------- |
| baseFare | Практически дублирует `totalFare` (отличается на сумму налогов и сборов). Если оставить оба признака, то при предсказании цены модель получит почти готовый ответ (утечка целевой переменной) |
| fareBasisCode | Код тарифа содержит очень много уникальных значений и фактически кодирует тип тарифа (в том числе базовую экономию), что приводит к утечке информации о классе билета |
| segmentsDepartureTimeEpochSeconds, segmentsArrivalTimeEpochSeconds | Дублируют информацию о времени вылета и прилёта в других форматах |
| segmentsArrivalTimeRaw | Время прилёта однозначно определяется временем вылета и длительностью перелёта |
| segmentsArrivalAirportCode, segmentsDepartureAirportCode | Содержат коды аэропортов пересадок (слишком много категорий); аэропорты вылета и прилёта уже есть в `startingAirport` и `destinationAirport` |
| segmentsAirlineCode | Дублирует `segmentsAirlineName` |
| segmentsEquipmentDescription | Содержит около сотни типов воздушных судов, т.е. слишком много категорий |
| segmentsDurationInSeconds, segmentsDistance | Дублируют `travelDuration` и `totalTravelDistance` на уровне отдельных сегментов |

In [8]:
df = df.select(
    "legId", "searchDate", "flightDate", "startingAirport", "destinationAirport",
    "travelDuration", "elapsedDays", "isBasicEconomy", "isRefundable", "isNonStop",
    "totalFare", "seatsRemaining", "totalTravelDistance",
    "segmentsDepartureTimeRaw", "segmentsAirlineName", "segmentsCabinCode"
)

In [9]:
df.show(5, truncate=30)

+------------------------------+----------+----------+---------------+------------------+--------------+-----------+--------------+------------+---------+---------+--------------+-------------------+-----------------------------+-------------------+-----------------+
|                         legId|searchDate|flightDate|startingAirport|destinationAirport|travelDuration|elapsedDays|isBasicEconomy|isRefundable|isNonStop|totalFare|seatsRemaining|totalTravelDistance|     segmentsDepartureTimeRaw|segmentsAirlineName|segmentsCabinCode|
+------------------------------+----------+----------+---------------+------------------+--------------+-----------+--------------+------------+---------+---------+--------------+-------------------+-----------------------------+-------------------+-----------------+
|9ca0e81111c683bec1012473fee...|2022-04-16|2022-04-17|            ATL|               BOS|       PT2H29M|          0|         False|       False|     True|   248.60|             9|                9

Выведем на экран метаданные датасета.

In [10]:
df.printSchema()

root
 |-- legId: string (nullable = true)
 |-- searchDate: string (nullable = true)
 |-- flightDate: string (nullable = true)
 |-- startingAirport: string (nullable = true)
 |-- destinationAirport: string (nullable = true)
 |-- travelDuration: string (nullable = true)
 |-- elapsedDays: string (nullable = true)
 |-- isBasicEconomy: string (nullable = true)
 |-- isRefundable: string (nullable = true)
 |-- isNonStop: string (nullable = true)
 |-- totalFare: string (nullable = true)
 |-- seatsRemaining: string (nullable = true)
 |-- totalTravelDistance: string (nullable = true)
 |-- segmentsDepartureTimeRaw: string (nullable = true)
 |-- segmentsAirlineName: string (nullable = true)
 |-- segmentsCabinCode: string (nullable = true)



Видно, что все столбцы датасета содержат строковый тип данных, что не соответствует ожиданиям. Кроме того, часть значений представлена в неудобном для анализа виде:
* длительность поездки записана в формате `ISO 8601` (например, `PT2H29M` означает 2 часа 29 минут);
* три столбца с информацией о сегментах перелёта содержат по несколько значений, склеенных через разделитель `||`.

Выполним преобразования типов данных и значений столбцов.

In [11]:
# Шаблон длительности в формате ISO 8601: P[n]DT[n]H[n]M
DURATION_PATTERN = r"^P(?:(\d+)D)?T(?:(\d+)H)?(?:(\d+)M)?$"


def duration_part(column: str, group_index: int):
    """
    Извлекает из строки длительности ISO 8601 одну из составляющих
    (дни, часы или минуты). Если составляющая отсутствует, возвращает 0.

    Args:
        column (str): Имя столбца со строкой длительности.
        group_index (int): Номер группы регулярного выражения
            (1 - дни, 2 - часы, 3 - минуты).

    Returns:
        Column: Числовое значение составляющей.
    """
    part = regexp_extract(col(column), DURATION_PATTERN, group_index)
    return when(part == "", lit(0)).otherwise(part.cast("Integer"))


def transform_dataframe(data: DataFrame) -> DataFrame:
    """
    Преобразует столбцы DataFrame в указанные типы данных и
    выполняет необходимые преобразования.

    Args:
        data (DataFrame): Исходный DataFrame.

    Returns:
        DataFrame: Преобразованный DataFrame.
    """
    # Преобразуем столбцы в соответствующие типы данных
    data = data.withColumn("searchDate",
                           to_date(col("searchDate"), "yyyy-MM-dd"))
    data = data.withColumn("flightDate",
                           to_date(col("flightDate"), "yyyy-MM-dd"))
    data = data.withColumn("elapsedDays",
                           col("elapsedDays").cast("Integer"))
    data = data.withColumn("isBasicEconomy",
                           col("isBasicEconomy").cast("Boolean"))
    data = data.withColumn("isRefundable",
                           col("isRefundable").cast("Boolean"))
    data = data.withColumn("isNonStop",
                           col("isNonStop").cast("Boolean"))
    data = data.withColumn("totalFare",
                           col("totalFare").cast("Float"))
    data = data.withColumn("seatsRemaining",
                           col("seatsRemaining").cast("Integer"))
    data = data.withColumn("totalTravelDistance",
                           col("totalTravelDistance").cast("Float"))

    # Переводим длительность из формата ISO 8601 в минуты
    data = data.withColumn(
        "travel_duration_min",
        when(col("travelDuration").rlike(DURATION_PATTERN),
             duration_part("travelDuration", 1) * 1440 +
             duration_part("travelDuration", 2) * 60 +
             duration_part("travelDuration", 3))
        .otherwise(lit(None).cast("Integer"))
    ).drop("travelDuration")

    # Преобразуем строки со значениями для сегментов в массивы строк
    for column in ["segmentsDepartureTimeRaw",
                   "segmentsAirlineName",
                   "segmentsCabinCode"]:
        data = data.withColumn(column, split(col(column), r"\|\|"))

    return data

In [12]:
df = transform_dataframe(df)

In [13]:
df.show(5, truncate=40)

+--------------------------------+----------+----------+---------------+------------------+-----------+--------------+------------+---------+---------+--------------+-------------------+-------------------------------+-------------------+-----------------+-------------------+
|                           legId|searchDate|flightDate|startingAirport|destinationAirport|elapsedDays|isBasicEconomy|isRefundable|isNonStop|totalFare|seatsRemaining|totalTravelDistance|       segmentsDepartureTimeRaw|segmentsAirlineName|segmentsCabinCode|travel_duration_min|
+--------------------------------+----------+----------+---------------+------------------+-----------+--------------+------------+---------+---------+--------------+-------------------+-------------------------------+-------------------+-----------------+-------------------+
|9ca0e81111c683bec1012473feefd28f|2022-04-16|2022-04-17|            ATL|               BOS|          0|         false|       false|     true|    248.6|             9|   

In [14]:
df.printSchema()

root
 |-- legId: string (nullable = true)
 |-- searchDate: date (nullable = true)
 |-- flightDate: date (nullable = true)
 |-- startingAirport: string (nullable = true)
 |-- destinationAirport: string (nullable = true)
 |-- elapsedDays: integer (nullable = true)
 |-- isBasicEconomy: boolean (nullable = true)
 |-- isRefundable: boolean (nullable = true)
 |-- isNonStop: boolean (nullable = true)
 |-- totalFare: float (nullable = true)
 |-- seatsRemaining: integer (nullable = true)
 |-- totalTravelDistance: float (nullable = true)
 |-- segmentsDepartureTimeRaw: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- segmentsAirlineName: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- segmentsCabinCode: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- travel_duration_min: integer (nullable = true)



Видно, что теперь столбцы датафрейма содержат значения корректных типов: даты, числа, логические значения и массивы.

Проверим, сколько строк вошло в датасет и какой период времени они охватывают: диапазон дат поиска (`searchDate`) должен совпасть с выбранным интервалом, а даты вылета (`flightDate`) лежат позже.

In [15]:
print(f"Количество строк: {df.count()}")

(
    df
    .select(
        min_("searchDate").alias("search_from"),
        max_("searchDate").alias("search_to"),
        min_("flightDate").alias("flight_from"),
        max_("flightDate").alias("flight_to")
    )
    .show()
)

Количество строк: 12857310


+-----------+----------+-----------+----------+
|search_from| search_to|flight_from| flight_to|
+-----------+----------+-----------+----------+
| 2022-04-16|2022-05-17| 2022-04-17|2022-07-12|
+-----------+----------+-----------+----------+



Видно, что после обрезки файла в датасете осталось 12 857 310 строк. Диапазон дат поиска (`searchDate`) полностью совпал с выбранным интервалом: с 2022-04-16 по 2022-05-17, то есть мы получили ровно один месяц наблюдений за ценами. Даты вылета (`flightDate`) лежат позже: с 2022-04-17 по 2022-07-12. Это логично: билеты, найденные в апреле и мае, предполагают вылет в течение ближайших двух месяцев. Соответственно, в данных будут представлены вылеты только в апреле–июле 2022 года.

Полученный датафрейм сохраним для дальнейшего использования. Сохранение выполним в таблицу `Apache Iceberg`. 

`Apache Iceberg` — это поддерживающий высокую производительность табличный формат для больших данных.

Сначала создадим базу данных, в которой будет расположена таблица. Во избежание путаницы, каждую таблицу следует называть с использованием фамилии студента.

In [16]:
# Замените ivanov на свою фамилию
database_name = "alieva_SAPR_database"

Создадим инструкцию SQL для добавления базы данных в каталог `Apache Spark`.

In [17]:
create_database_sql = f"""
CREATE DATABASE IF NOT EXISTS spark_catalog.{database_name}
"""

In [18]:
spark.sql(create_database_sql)

DataFrame[]

Установим созданную базу данных как текущую.

In [19]:
spark.catalog.setCurrentDatabase(database_name)

И, наконец, записываем преобразованный датафрейм в таблицу `sobd_lab1_table`.

In [20]:
# Сохранение DataFrame в виде таблицы
df.writeTo("sobd_lab1_table").using("iceberg").createOrReplace()

После успешной записи можно посмотреть, какие таблицы входят в базу данных.

In [21]:
for table in spark.catalog.listTables():
    print(table.name)

sobd_lab1_table


Видно, что таблица `sobd_lab1_table` создана в базе данных пользователя. Именно её мы прочитаем во второй части лабораторной работы для разведочного анализа и очистки данных.

Обратите внимание, что при необходимости созданные базу данных и таблицу можно удалить следующими командами.

In [22]:
# spark.sql("DROP TABLE spark_catalog.alieva_SAPR_database.sobd_lab1_table")
# spark.sql("DROP DATABASE spark_catalog.alieva_SAPR_database")

После успешной записи таблицы останавливаем сессию `Apache Spark`.

In [23]:
spark.stop()